In [ ]:
%%writefile s6e10_mlp.py
#!/usr/bin/env python3
"""
S6E10 MLP STUDENT with soft labels (teacher = your stack's OOF + the 0.96213 submission as test pseudo-labeller).
Same idea as s6e10_pseudo.py (XGBoost) but the student is a PyTorch MLP: rank-gauss numerics + embeddings (categoricals, ratings,
route key), BCE with soft targets and per-row weights, OneCycle AdamW, best epoch on the held-out fold, K folds x seeds.
  train rows : target alpha*y + (1-alpha)*p_teacher_oof, weight 1
  test rows  : target p_teacher_test (the CSV quantile-mapped onto your stack's probability distribution), weight lam
RESULT OF THE XGBOOST STUDENT (your run): OOF 0.96075 -> 0.96225 as lam 0 -> 3, but every submission (0.7 teacher + 0.3 student) scored
  0.96195-0.96204 < 0.96213 (teacher alone), ordered by lam: more teacher in the student = closer to the teacher, not past it. The OOF
  rise is mostly the student learning the teacher plus leakage (the teacher saw all train labels). A student only helps the LB if it
  carries information the teacher lacks: an MLP's different inductive bias is the realistic source, so judge it by the blend weights
  below (w=0.3/0.5/1) and, better, by adding mlp_oof_test.npz to your own stack.
Tested on synthetic data with CPU torch only. Run: !python s6e10_mlp.py --base <lite_oof_test.npz> --teacher-csv <0.96213.csv> [--quick]
Outputs: mlp_sub_a<alpha>_l<lam>_w<w>.csv (sigmoid((1-w)*z(teacher)+w*z(student))), mlp_oof_test.npz, mlp_summary.csv.
"""
import argparse
import glob
import hashlib
import math
import os
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import xgboost as xgb
from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

warnings.filterwarnings("ignore")
T0 = time.time()
TARGET = "satisfaction"
SEED = 42


def log(m):
    print(f"[{time.time() - T0:5.0f}s] {m}", flush=True)


STEP = [0]


def step(m):
    STEP[0] += 1
    print(f"\n[{time.time() - T0:5.0f}s] ===== STEP {STEP[0]}: {m} =====", flush=True)


class OutOfTime(Exception):
    pass


def args_():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="")
    p.add_argument("--og", default="")
    p.add_argument("--base", required=True, help="your stack: npz with oof/te (+ w, b)")
    p.add_argument("--teacher-csv", default="", help="submission csv used as test pseudo-labeller (quantile-mapped onto your stack)")
    p.add_argument("--variants", default="1,3", help="alpha,lam pairs separated by ;")
    p.add_argument("--temp", type=float, default=1.0)
    p.add_argument("--folds", type=int, default=10)
    p.add_argument("--seeds", type=int, default=2)
    p.add_argument("--epochs", type=int, default=14)
    p.add_argument("--bs", type=int, default=4096)
    p.add_argument("--lr", type=float, default=3e-3)
    p.add_argument("--blend-w", default="0.3,0.5,1", help="student weights for the submissions")
    p.add_argument("--rows", type=int, default=0)
    p.add_argument("--quick", action="store_true")
    p.add_argument("--cpu", action="store_true")
    p.add_argument("--out", default="/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    return p.parse_args()


# ------------------------------------------------------------------------------------------------ data (verbatim from s6e10_lite.py)
def norm(s):
    return str(s).strip().lower()


def find_dirs(a):
    comp = a.data
    if not comp:
        c = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)]
        c = [d for d in c if os.path.exists(d + "/train.csv") and os.path.exists(d + "/test.csv")]
        comp = sorted(c, key=lambda d: "s6e10" not in d)[0]
    og = a.og
    if og != "none" and not og:
        c = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)]
        c = [d for d in c if "airline" in d.lower() and "playground" not in d.lower()]
        og = c[0] if c else ""
    return comp, ("" if og == "none" else og)


def load(a):
    comp, ogd = find_dirs(a)
    tr, te = pd.read_csv(comp + "/train.csv"), pd.read_csv(comp + "/test.csv")
    sub = pd.read_csv(comp + "/sample_submission.csv") if os.path.exists(comp + "/sample_submission.csv") else None
    y = tr[TARGET].astype(str).str.lower().isin(["true", "1", "1.0", "satisfied"]).to_numpy().astype(int)
    cols = [c for c in te.columns if norm(c) != "id"]
    idc = [c for c in te.columns if norm(c) == "id"][0]
    og_x, og_y = None, None
    if ogd:
        try:
            parts = [pd.read_csv(f"{ogd}/{f}") for f in ("train.csv", "test.csv") if os.path.exists(f"{ogd}/{f}")]
            og = pd.concat(parts, ignore_index=True)
            og.columns = [c.strip() for c in og.columns]
            m = {norm(c): c for c in og.columns}
            if all(norm(c) in m for c in cols) and "satisfaction" in m:
                og_x = og[[m[norm(c)] for c in cols]].copy()
                og_x.columns = cols
                og_y = og[m["satisfaction"]].astype(str).str.lower().isin(["satisfied", "true", "1"]).to_numpy().astype(int)
                keep = ~og_x.assign(_y=og_y).duplicated().to_numpy()
                og_x, og_y = og_x[keep].reset_index(drop=True), og_y[keep]
        except Exception as e:
            log(f"original data not usable: {e!r}")
    log(f"train={len(tr):,} test={len(te):,} features={len(cols)} positive rate={y.mean():.4f} | "
        f"original data: {'none' if og_x is None else f'{len(og_x):,} rows'}")
    return tr[cols], te[cols], y, te[idc].to_numpy(), idc, sub, og_x, og_y, cols


def encode(tr_x, te_x, og_x):
    """Everything to float32. Strings -> integer codes shared across all frames. Returns frames + metadata."""
    frames = [tr_x, te_x] + ([og_x] if og_x is not None else [])
    ALL = pd.concat([f.reset_index(drop=True) for f in frames], ignore_index=True)
    cat_cols = [c for c in ALL.columns if not pd.api.types.is_numeric_dtype(ALL[c])]   # strings (object or str dtype)
    for c in ALL.columns:
        if c in cat_cols:
            v = ALL[c].astype(str).str.strip().str.lower()
            ALL[c] = pd.Categorical(v, categories=sorted(v.unique())).codes.astype(np.float32)
        else:
            v = pd.to_numeric(ALL[c], errors="coerce")
            ALL[c] = v.fillna(v.median()).astype(np.float32)
    n1, n2 = len(tr_x), len(te_x)
    parts = [ALL.iloc[:n1], ALL.iloc[n1:n1 + n2]] + ([ALL.iloc[n1 + n2:]] if og_x is not None else [])
    parts = [p.reset_index(drop=True) for p in parts]
    ratings = [c for c in ALL.columns if c not in cat_cols and ALL[c].nunique() <= 7 and ALL[c].min() >= 0 and ALL[c].max() <= 5]
    low = {c: norm(c) for c in ALL.columns}
    pick = lambda *keys: [c for c in ALL.columns if any(k in low[c] for k in keys)]
    meta = dict(cat=cat_cols, ratings=ratings, fd=pick("distance")[0], age=[c for c in ALL.columns if low[c] == "age"][0],
                dep=[c for c in pick("delay") if "depart" in low[c]][0], arr=[c for c in pick("delay") if "arriv" in low[c]][0],
                bg=[c for k in ("class", "type of travel", "customer type") for c in pick(k)][:3])
    return parts, meta


# ------------------------------------------------------------------------------------------------ features (verbatim)
def basic(X, meta, pairs=None):
    """Row-level features: raw columns + zero-count + non-zero mean (+ zero-pair flags) + delay difference."""
    F = X.copy()
    R = X[meta["ratings"]].to_numpy()
    Z = R == 0
    F["zero_count"] = Z.sum(1).astype(np.float32)
    nz = (~Z).sum(1)
    F["nonzero_mean"] = np.where(nz > 0, (R * (~Z)).sum(1) / np.maximum(nz, 1), 0).astype(np.float32)
    F["delay_diff"] = X[meta["arr"]] - X[meta["dep"]]
    return F


def route_block(F, meta):
    """Label-free route context: Flight Distance = route id. Computed on train+test together."""
    key = F[meta["fd"]].round().astype(np.int64)
    cols = meta["ratings"] + ["zero_count", "nonzero_mean", meta["age"], meta["dep"], meta["arr"]] + meta["bg"]
    G = F[cols].groupby(key).transform("mean").astype(np.float32)
    G.columns = ["rt_" + c for c in cols]
    G["rt_count"] = key.map(key.value_counts()).astype(np.float32)
    return G


def xgb_params(gpu, depth, eta, seed=SEED):
    p = dict(objective="binary:logistic", eval_metric="auc", eta=eta, max_depth=depth, subsample=0.8, colsample_bytree=0.5,
             min_child_weight=5, reg_lambda=2.0, tree_method="hist", seed=seed, max_bin=256)
    p["device"] = "cuda" if gpu else "cpu"
    if not gpu:
        p["nthread"] = os.cpu_count() or 1
    return p


def og_feature(F_all, F_og, y_og, y_tr, n_tr, meta, gpu):
    """Model trained ONLY on the original data, scored on every competition row (no competition label used)."""
    d = xgb.DMatrix(F_og.to_numpy(np.float32), y_og)
    b = xgb.train(xgb_params(gpu, 6, 0.1), d, 300)
    m = b.predict(xgb.DMatrix(F_all.to_numpy(np.float32)), output_margin=True)
    log(f"original-only xgb: AUC on competition labels {roc_auc_score(y_tr, m[:n_tr]):.5f} (expect ~0.955); margin std {m.std():.2f}")
    key = F_all[meta["fd"]].round().astype(np.int64)
    out = pd.DataFrame({"og_xgb": m.astype(np.float32)})
    out["og_route_mean"] = pd.Series(m).groupby(key.to_numpy()).transform("mean").astype(np.float32)
    return out


def stack(Z, y):
    """Non-negative logistic stacker on member logits (>= 0 weights stop collinear members cancelling out)."""
    Z = Z.astype(np.float64)

    def f(p):
        s = Z @ p[:-1] + p[-1]
        r = expit(s) - y
        return np.mean(np.logaddexp(0, s) - y * s), np.append(Z.T @ r, r.sum()) / len(y)
    p0 = np.append(np.full(Z.shape[1], 1.0 / Z.shape[1]), 0.0)
    r = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=[(0, None)] * Z.shape[1] + [(None, None)])
    return r.x[:-1], r.x[-1]


# ================================================================================================ MLP STUDENT
def zs(v):
    return (v - v.mean()) / (v.std() + 1e-12)


def load_teacher(path, n_full, n_te, temp):
    z = np.load(path, allow_pickle=True)
    oof, te = np.asarray(z["oof"], np.float64), np.asarray(z["te"], np.float64)
    if oof.ndim == 2:
        if "w" in z.files:
            w, b = np.asarray(z["w"], np.float64), float(np.asarray(z["b"]))
            oof, te = oof @ w + b, te @ w + b
        else:
            oof, te = oof.mean(1), te.mean(1)
    if len(oof) != n_full or len(te) != n_te:
        sys.exit(f"--base has {len(oof):,}/{len(te):,} rows, expected {n_full:,}/{n_te:,}")
    if oof.min() >= 0 and oof.max() <= 1:
        oof, te = logit(np.clip(oof, 1e-6, 1 - 1e-6)), logit(np.clip(te, 1e-6, 1 - 1e-6))
    return oof, te, expit(oof / temp), expit(te / temp)


class Net(nn.Module):
    def __init__(s, card, n_num, hid=(512, 256, 128), drop=0.15):
        super().__init__()
        s.emb = nn.ModuleList([nn.Embedding(c, 16 if c > 50 else 4) for c in card])
        d = n_num + sum(e.embedding_dim for e in s.emb)
        L = []
        for h in hid:
            L += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(drop)]
            d = h
        s.mlp = nn.Sequential(*L, nn.Linear(d, 1))

    def forward(s, xn, xc):
        return s.mlp(torch.cat([xn] + [e(xc[:, j]) for j, e in enumerate(s.emb)], 1)).squeeze(1)


@torch.no_grad()
def predict(net, Tn, Tc, idx):
    net.eval()
    return np.concatenate([net(Tn[idx[i:i + 16384]], Tc[idx[i:i + 16384]]).float().cpu().numpy() for i in range(0, len(idx), 16384)])


def fit_fold(Tn, Tc, card, y, tgt, wgt, a, b, n, lam, args, dev, seed):
    """a/b: train-row indices (train / held-out). Test rows are rows n.. of Tn. Early stop on held-out AUC with the true labels."""
    torch.manual_seed(seed)
    N = Tn.shape[0]
    ia = torch.tensor(np.concatenate([a, np.arange(n, N)]) if lam > 0 else a, device=dev)
    ib, it_ = torch.tensor(b, device=dev), torch.arange(n, N, device=dev)
    net = Net(card, Tn.shape[1]).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=args.lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=args.epochs * math.ceil(len(ia) / args.bs), pct_start=0.2)
    best = (-1.0, None, -1)
    for ep in range(args.epochs):
        net.train()
        perm = ia[torch.randperm(len(ia), device=dev)]
        for i in range(0, len(perm), args.bs):
            j = perm[i:i + args.bs]
            if len(j) < 2:
                continue
            l = (F.binary_cross_entropy_with_logits(net(Tn[j], Tc[j]), tgt[j], reduction="none") * wgt[j]).sum() / wgt[j].sum()
            opt.zero_grad(set_to_none=True)
            l.backward()
            opt.step()
            sch.step()
        auc = roc_auc_score(y[b], predict(net, Tn, Tc, ib))
        if auc > best[0]:
            best = (auc, {k: v.clone() for k, v in net.state_dict().items()}, ep)
    net.load_state_dict(best[1])
    return predict(net, Tn, Tc, ib), predict(net, Tn, Tc, it_), best[0], best[2] + 1


def main():
    args = args_()
    if args.quick:
        args.rows, args.folds, args.seeds, args.epochs = args.rows or 60000, 3, 1, 3
    dev = torch.device("cuda" if torch.cuda.is_available() and not args.cpu else "cpu")
    os.makedirs(args.out, exist_ok=True)
    variants = [tuple(float(v) for v in s.split(",")) for s in args.variants.split(";") if s.strip()]
    log(f"MLP student: device={dev} folds={args.folds} seeds={args.seeds} epochs={args.epochs} variants={variants}")

    step("loading data")
    tr_x, te_x, y, ids, idc, sub, og_x, og_y, cols = load(args)
    n_full, keep = len(y), None
    if args.rows and args.rows < n_full:
        keep = np.random.RandomState(SEED).choice(n_full, args.rows, replace=False)
        tr_x, y = tr_x.iloc[keep].reset_index(drop=True), y[keep]
    n, n_te = len(tr_x), len(te_x)
    t_oof, t_te, p_oof, p_te = load_teacher(args.base, n_full, n_te, args.temp)
    if keep is not None:
        t_oof, p_oof = t_oof[keep], p_oof[keep]
    Tt_logit = t_te
    if args.teacher_csv:
        tc = pd.read_csv(args.teacher_csv)
        s = tc.set_index(tc.columns[0]).iloc[:, 0].reindex(ids).to_numpy(np.float64)
        if np.isnan(s).any():
            sys.exit("--teacher-csv does not cover the test ids")
        p_te = np.sort(p_te)[rankdata(s, method="ordinal").astype(int) - 1]
        Tt_logit = logit(np.clip(p_te, 1e-6, 1 - 1e-6)) * args.temp
    log(f"your stack OOF AUC {roc_auc_score(y, t_oof):.5f}; test pseudo-label mean p {p_te.mean():.4f}")

    step("features -> rank-gauss numerics + embedding codes")
    parts, meta = encode(tr_x, te_x, og_x)
    Fall = basic(pd.concat(parts[:2], ignore_index=True), meta)
    blocks = [Fall, route_block(Fall, meta)]
    if og_x is not None:
        blocks.append(og_feature(Fall, basic(parts[2], meta), og_y, y, n, meta, torch.cuda.is_available() and not args.cpu))
    Fdf = pd.concat(blocks, axis=1)
    num = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=2_000_000, random_state=SEED).fit_transform(Fdf.to_numpy(np.float32)).astype(np.float32)
    ec = [c for c in meta["cat"] + meta["ratings"] + [meta["fd"]] if c in Fdf.columns]
    codes = np.column_stack([pd.factorize(Fdf[c])[0] for c in ec]).astype(np.int64)
    card = [int(codes[:, j].max()) + 1 for j in range(codes.shape[1])]
    Tn, Tc = torch.tensor(num, device=dev), torch.tensor(codes, device=dev)
    log(f"inputs: {num.shape[1]} numeric, {len(card)} embedded (largest {max(card)} levels)")
    folds = list(StratifiedKFold(args.folds, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    Zt = zs(t_oof)

    step("training")
    res, names, O, T_ = [], [], [], []
    for alpha, lam in variants:
        name = f"alpha={alpha:g},lam={lam:g}"
        tgt = torch.tensor(np.concatenate([alpha * y + (1 - alpha) * p_oof, p_te]), dtype=torch.float32, device=dev)
        wgt = torch.tensor(np.concatenate([np.ones(n), np.full(n_te, lam)]), dtype=torch.float32, device=dev)
        oof, te, aucs = np.zeros(n), np.zeros(n_te), []
        for sd in range(args.seeds):
            for k, (a, b) in enumerate(folds):
                t = time.time()
                pv, pt, auc, ep = fit_fold(Tn, Tc, card, y, tgt, wgt, a, b, n, lam, args, dev, SEED + 100 * sd + k)
                aucs.append(auc)
                oof[b] += zs(pv) / args.seeds
                te += zs(pt) / (len(folds) * args.seeds)
                log(f"  {name} seed {sd} fold {k + 1}/{len(folds)} AUC {auc:.5f} (best epoch {ep}, {time.time() - t:.0f}s)")
        te = zs(te)
        w, b0 = stack(np.column_stack([Zt, oof]), y)
        sa = roc_auc_score(y, np.column_stack([Zt, oof]) @ w + b0)
        res.append(dict(variant=name, mean_fold_auc=float(np.mean(aucs)), pooled_auc=roc_auc_score(y, oof), stack_with_teacher=sa,
                        stack_gain=sa - roc_auc_score(y, Zt), spearman_vs_teacher=pd.Series(Zt).corr(pd.Series(oof), method="spearman")))
        names.append((alpha, lam)); O.append(oof); T_.append(te)
        log(f"  => {name}: mean fold AUC {res[-1]['mean_fold_auc']:.5f} | pooled {res[-1]['pooled_auc']:.5f} | stack gain over your stack {res[-1]['stack_gain']:+.5f} | Spearman {res[-1]['spearman_vs_teacher']:.4f}")
    df = pd.DataFrame(res)
    df.to_csv(os.path.join(args.out, "mlp_summary.csv"), index=False)
    log("OOF numbers of soft variants are leaky (teacher saw all train labels); the public LB decides.\n" + df.round(5).to_string(index=False))

    step("outputs")
    Tt = zs(Tt_logit)
    for (alpha, lam), te_v in zip(names, T_):
        for w in [float(x) for x in args.blend_w.split(",")]:
            pd.DataFrame({(sub.columns[0] if sub is not None else idc): ids, (sub.columns[1] if sub is not None else TARGET): expit((1 - w) * Tt + w * te_v)}) \
                .to_csv(os.path.join(args.out, f"mlp_sub_a{alpha:g}_l{lam:g}_w{w:g}.csv"), index=False)
    np.savez(os.path.join(args.out, "mlp_oof_test.npz"), names=np.array([f"alpha={a:g},lam={l:g}" for a, l in names]), oof=np.column_stack(O), te=np.column_stack(T_))
    log(f"done; total {(time.time() - T0) / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
ARGS = "--base /kaggle/input/datasets/yukrxfo/airplane/oof_test_logits.npz --teacher-csv /kaggle/input/datasets/yukrxfo/airplane/0.96213.csv"   # add --quick first for a minutes-long check
!python s6e10_mlp.py $ARGS
